In [1]:
print("OK")

OK


In [4]:
import os 
#moving one folder up

os.chdir("../")

In [5]:
%pwd

'c:\\Users\\DELL\\Desktop'

In [6]:
from langchain.document_loaders import PyPDFLoader
from langchain_google_genai import GoogleGenerativeAI
from dotenv import load_dotenv

In [8]:
load_dotenv()

False

In [9]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [12]:
#extract text from PDF files
def load_pdf_files(data):
    loader=PyPDFLoader(data)
    documents=loader.load()
    return documents

In [17]:
%pwd

'c:\\Users\\DELL\\Desktop'

In [18]:
extracted_data=load_pdf_files("RAG-BASED-MEDICAL-CHATBOT/data/Medical_book.pdf")

In [23]:
extracted_data[100]

Document(metadata={'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'creator': 'PyPDF', 'creationdate': '2004-12-18T17:16:32-05:00', 'moddate': '2004-12-18T16:35:04-06:00', 'source': 'RAG-BASED-MEDICAL-CHATBOT/data/Medical_book.pdf', 'total_pages': 759, 'page': 100, 'page_label': '101'}, page_content='cancer, the risk factor begins to increase in the late teens.\nRates for carcinoma in situ peak between the ages of 20\nand 30. In the United States, the incidence of invasive\ncervical cancer increases rapidly with age for African\nAmerican women over the age of 25. The incidence rises\nmore slowly for Caucasian women. However, women\nover age 65 account for more than 25% of all cases of\ninvasive cervical cancer.\nThe incidence of cervical cancer is highest among\npoor women and among women in developing countries.\nIn the United States, the death rates from cervical cancer\nare higher among Hispanic, Native American, and\nAfrican American women than among Caucasian\nwomen. These groups of women

In [22]:
#one for each page 
len(extracted_data)

759

In [ ]:
from typing import List
from langchain.schema import Document

#only the page_content and the source is important 
def filter_to_minimal_docs(docs:List[Document])->List[Document]:
    minimal_docs:List[Document]=[]
    for doc in docs:
        src=doc.metadata.get("source")
        minimal_docs.append(
            Document(
            page_content=doc.page_content,
            metadata={"source":src}
        ))
    return minimal_docs

In [25]:
minimal_docs=filter_to_minimal_docs(extracted_data)

In [28]:
minimal_docs[150]

Document(metadata={'source': 'RAG-BASED-MEDICAL-CHATBOT/data/Medical_book.pdf'}, page_content='• Hepatitis B. Three doses.\n• Diphtheria,Tetanus, and Pertussis (DTaP). Four doses.\n• H. influenzae type b (Hib). Four doses.\n• Inactivated Polio. Three doses.\n• Pneumococcal Conjugate. Three doses.\n• Measles,Mumps, Rubella (MMR). One dose.\n• Varicella (chickenpox). One dose.\n• Hepatitis A. (In certain geographical areas and with\ncertain high risk groups.)\nSome immunizations may cause mild side effects, or\nmore rarely, serious adverse reactions. However, the ben-\nefits of immunization greatly outweigh the incidence of\nhealth problems arising from them.\nThere are serious chronic diseases and health prob-\nlems that are frequently diagnosed in childhood and can-\nnot be vaccinated against. These include, but are not lim-\nited to, asthma, type I diabetes (juvenile diabetes),\nleukemia, hemophilia, and cystic fibrosis.\nMental health\nChildren who have difficulty in areas of languag

In [29]:
#we will do the chunking 
def text_split(minimal_docs):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )
    texts_chunks=text_splitter.split_documents(minimal_docs)
    return texts_chunks

In [30]:
texts_chunk=text_split(minimal_docs)
print(f"Number of Chunks:{len(texts_chunk)}")

Number of Chunks:4138


In [31]:
texts_chunk[0]

Document(metadata={'source': 'RAG-BASED-MEDICAL-CHATBOT/data/Medical_book.pdf'}, page_content='The GALE\nENCYCLOPEDIA\nof MEDICINE\nSECOND EDITION')

In [36]:
from langchain.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    # Repo ID for the model we discussed (80MB)
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    
  

    embeddings = HuggingFaceEmbeddings(
        model_name=model_name,
        
    )
    return embeddings


    

In [40]:
embedding=download_embeddings()

c:\Users\DELL\Desktop\RAG-BASED-MEDICAL-CHATBOT\medibot\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\DELL\Desktop\RAG-BASED-MEDICAL-CHATBOT\medibot\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\DELL\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to r

In [41]:
vectors=embedding.embed_query("What is the capital of France?")


In [42]:
vectors

[0.08204812556505203,
 0.0360555574297905,
 -0.0038928636349737644,
 -0.004881070926785469,
 0.0256511140614748,
 -0.05714341625571251,
 0.012191600166261196,
 0.004678933881223202,
 0.03494987636804581,
 -0.02242186665534973,
 -0.008005287498235703,
 -0.10935358703136444,
 0.022724727168679237,
 -0.02932085283100605,
 -0.04352201893925667,
 -0.1202412024140358,
 -0.0008486219448968768,
 -0.01815015636384487,
 0.056129541248083115,
 0.003085241885855794,
 0.002336402190849185,
 -0.01683925837278366,
 0.06362467259168625,
 -0.023660244420170784,
 0.03149351850152016,
 -0.03479795902967453,
 -0.02054879628121853,
 -0.0027910363860428333,
 -0.0110379783436656,
 -0.03612669184803963,
 0.0541410818696022,
 -0.036617107689380646,
 -0.02500867284834385,
 -0.03817036747932434,
 -0.0496036522090435,
 -0.01514811534434557,
 0.021315036341547966,
 -0.01274044718593359,
 0.07670092582702637,
 0.04435577988624573,
 -0.010834827087819576,
 -0.02975994348526001,
 -0.016970515251159668,
 -0.0246918275

In [43]:
print(len(vectors))

384


In [46]:
%pwd

'c:\\Users\\DELL\\Desktop'

In [47]:
import os 
os.chdir("RAG-BASED-MEDICAL-CHATBOT")

In [48]:
load_dotenv()

True

In [60]:
pinecone_api_key = os.getenv("PINECONE_ENV")

In [67]:
pinecone_api_key

'pcsk_3ZytC3_R4C1NibvJzVmvFEiL9NskEk28iF1x1zwJTV4TBejo9gqEHTryQ6x5kZfJvxjjGc'

In [61]:
from pinecone import Pinecone


pc=Pinecone(api_key=pinecone_api_key)

In [62]:
pc

In [63]:
from pinecone import ServerlessSpec
index_name="medical-chatbot"
if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,#embedding function dimensions 
        metric="cosine",#for cosine similarity
        spec=ServerlessSpec(cloud="aws",region="us-east-1")
    )
index=pc.Index(index_name)

In [64]:
index

In [71]:
import os
from dotenv import load_dotenv
from langchain_pinecone import PineconeVectorStore

# 1. Load your .env file
load_dotenv()

# 2. Safely retrieve the key
pinecone_api_key = os.getenv("PINECONE_API_KEY")

# 3. Check if it's missing to avoid the "str | None" error
if pinecone_api_key is None:
    raise ValueError("PINECONE_API_KEY is not set in your .env file or the file wasn't found.")

# 4. Set the environment variable as a string
os.environ["PINECONE_API_KEY"] = pinecone_api_key

# 5. Initialize the Vector Store
vs = PineconeVectorStore.from_documents(
    documents=texts_chunk,
    embedding=embedding,
    index_name=index_name
)


In [72]:
#LOADING THE EXISTING INDEX
docsearch=PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
)

In [73]:
#adding more data to an existing index
dswith=Document(
    page_content="an youtube channel that provides tutorial for jee neet ",
    metadata={"source":"Youtube"}
)

In [74]:
docsearch.add_documents([dswith])

['64103ea0-fb60-43fd-8d6f-f39655c9b7b4']

In [75]:
retriever=docsearch.as_retriever(search_type="similarity",search_kwargs={"k":3})

In [76]:
respone=retriever.invoke("What is Acne?")

In [77]:
print(respone)

[Document(id='4b505466-142d-4a24-8031-2d09ed5c3a77', metadata={'source': 'RAG-BASED-MEDICAL-CHATBOT/data/Medical_book.pdf'}, page_content='National Institute of Mental Health. Mental Health Public\nInquiries, 5600 Fishers Lane, Room 15C-05, Rockville,\nMD 20857. (888) 826-9438. <http://www.nimh.nih.gov>.\nPaula Anne Ford-Martin\nDermabrasion see Skin resurfacing\nDermatitis\nDefinition\nDermatitis is a general term used to describe inflam-\nmation of the skin.\nDescription\nMost types of dermatitis are characterized by an\nitchy pink or red rash.\nContact dermatitis is an allergic reaction to some-\nthing that irritates the skin and is manifested by one or\nmore lines of red, swollen, blistered skin that may itch or\nGALE ENCYCLOPEDIA OF MEDICINE 21036\nDermatitis'), Document(id='8fe9e2c8-8e6d-4add-95d1-0a402339fcf4', metadata={'source': 'RAG-BASED-MEDICAL-CHATBOT/data/Medical_book.pdf'}, page_content='tion and relieve irritation. Creams, lotions, or ointments\nnot specifically formula

In [80]:
load_dotenv()

True

In [81]:
from langchain_google_genai import ChatGoogleGenerativeAI
model=ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [82]:
from langchain_core.prompts import ChatPromptTemplate


In [83]:
system_prompt=(
    "you are an assistant for question-answering tasks."
    "Use the Following pieces of retreived context to answer "
    "the question .If you don't know the answer ,say that you"
    "dont know .Use three sentences maximum and keep the answer concise"
    "\n\n"
    "{context}"
)
prompt=ChatPromptTemplate.from_messages(
    [
        ("system",system_prompt),
        ("human","{input}"),
    ]
)

In [85]:
from langchain_core.output_parsers import StrOutputParser
parser=StrOutputParser()

In [87]:
#for avoiding the manuial chains linkign we have now 
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain

In [88]:
question_answer_chain=create_stuff_documents_chain(model,prompt)
rag_chain=create_retrieval_chain(retriever,question_answer_chain)


In [96]:
response=rag_chain.invoke({
    "input":"What is cancer"
})

In [97]:
print(response["answer"])

Cancer is defined as a disease of the genes, characterized by abnormal cells that divide uncontrollably to form a new growth called a tumor or neoplasm. Specifically, a malignant tumor is considered cancer, which invades surrounding tissue and spreads to other parts of the body. These abnormal cells arise due to mutations in DNA, often caused by environmental factors called carcinogens.
